In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load gap scores data from Phase 3
df = pd.read_csv('../data/outputs/gap_scores.csv')
print(f"✅ Loaded! {df.shape[0]} rows × {df.shape[1]} columns")

✅ Loaded! 19548 rows × 32 columns


In [2]:
# How engaged are YouTube viewers?
# A song with 1M views and 500K likes is more engaging
# than a song with 10M views and 100K likes

df['engagement_ratio'] = df['Likes'] / (df['Views'] + 1)
# +1 avoids division by zero

print("✅ Engagement Ratio created!")
print(df['engagement_ratio'].describe())

✅ Engagement Ratio created!
count    19548.000000
mean         0.012226
std          0.011239
min          0.000000
25%          0.005675
50%          0.008785
75%          0.015006
max          0.249204
Name: engagement_ratio, dtype: float64


In [3]:
# How many streams does each like generate?
# High value = people stream but don't engage on YouTube

df['stream_per_like'] = df['Stream'] / (df['Likes'] + 1)

print("✅ Stream per Like created!")
print(df['stream_per_like'].describe())

✅ Stream per Like created!
count    1.954800e+04
mean     8.613813e+04
std      7.229398e+06
min      6.029304e-03
25%      1.494780e+02
50%      3.573992e+02
75%      1.193396e+03
max      1.004576e+09
Name: stream_per_like, dtype: float64


In [4]:
# Convert milliseconds to minutes — more interpretable
df['duration_min'] = df['Duration_ms'] / 60000

# Categorise song length
def duration_category(mins):
    if mins < 2:
        return 'very_short'
    elif mins < 3.5:
        return 'short'
    elif mins < 5:
        return 'standard'
    else:
        return 'long'

df['duration_category'] = df['duration_min'].apply(duration_category)

print("✅ Duration features created!")
print(df['duration_category'].value_counts())

✅ Duration features created!
duration_category
short         8642
standard      8403
long          1905
very_short     598
Name: count, dtype: int64


In [5]:
# Songs that are both energetic AND danceable tend to perform differently
# This combined feature captures that interaction

df['energy_dance'] = df['Energy'] * df['Danceability']

print("✅ Energy-Dance interaction created!")
print(df['energy_dance'].describe())

✅ Energy-Dance interaction created!
count    19548.000000
mean         0.402687
std          0.169207
min          0.000000
25%          0.289856
50%          0.417600
75%          0.525430
max          0.880780
Name: energy_dance, dtype: float64


In [6]:
# Combine Valence (happiness) and Energy into one positivity score
# High positivity = upbeat songs

df['positivity_score'] = (df['Valence'] + df['Energy']) / 2

print("✅ Positivity Score created!")
print(df['positivity_score'].describe())

✅ Positivity Score created!
count    19548.000000
mean         0.581783
std          0.191634
min          0.000015
25%          0.460500
50%          0.605000
75%          0.725500
max          0.981000
Name: positivity_score, dtype: float64


In [7]:
# For each artist — what is their average gap score?
# Positive avg = they are a Spotify artist
# Negative avg = they are a YouTube artist

artist_identity = df.groupby('Artist')['gap_score'].mean().reset_index()
artist_identity.columns = ['Artist', 'artist_platform_score']

# Merge back into main dataframe
df = df.merge(artist_identity, on='Artist', how='left')

print("✅ Artist Platform Identity created!")
print(df['artist_platform_score'].describe())

✅ Artist Platform Identity created!
count    19548.000000
mean         1.570714
std          1.382343
min         -3.520290
25%          0.592968
50%          1.495553
75%          2.405186
max         10.681342
Name: artist_platform_score, dtype: float64


In [8]:
# ML models only understand numbers — not text
# So we convert text columns to numbers

# Album type → numbers
df['album_type_encoded'] = df['Album_type'].map({
    'album': 0,
    'single': 1,
    'compilation': 2
})

# Platform class → numbers (this is our target for classification)
df['platform_label'] = df['platform_class'].map({
    'YouTube-native': 0,
    'Cross-platform': 1,
    'Spotify-native': 2
})

print("✅ Categorical columns encoded!")
print(df['platform_label'].value_counts())

✅ Categorical columns encoded!
platform_label
2    6646
0    6451
1    6451
Name: count, dtype: int64


In [9]:
# All the new features we created
new_features = [
    'gap_score', 'log_streams', 'log_views',
    'engagement_ratio', 'stream_per_like',
    'duration_min', 'energy_dance',
    'positivity_score', 'artist_platform_score',
    'album_type_encoded', 'platform_label'
]

print("✅ New features summary:")
print(df[new_features].describe().round(3))

✅ New features summary:
       gap_score  log_streams  log_views  engagement_ratio  stream_per_like  \
count  19548.000    19548.000  19548.000         19548.000     1.954800e+04   
mean       1.571       17.648     16.077             0.012     8.613813e+04   
std        2.224        1.649      2.766             0.011     7.229398e+06   
min       -9.511        8.791      3.296             0.000     6.000000e-03   
25%        0.127       16.697     14.457             0.006     1.494780e+02   
50%        1.151       17.727     16.508             0.009     3.573990e+02   
75%        2.574       18.755     18.084             0.015     1.193396e+03   
max       17.432       21.943     22.813             0.249     1.004576e+09   

       duration_min  energy_dance  positivity_score  artist_platform_score  \
count     19548.000     19548.000         19548.000              19548.000   
mean          3.742         0.403             0.582                  1.571   
std           2.114         0.

In [10]:
# Save to processed folder for ML phases
df.to_csv('../data/processed/featured.csv', index=False)

print(f"✅ Featured data saved to data/processed/featured.csv")
print(f"Total columns now: {df.shape[1]}")

✅ Featured data saved to data/processed/featured.csv
Total columns now: 41


In [12]:
!pip install xgboost scikit-learn